In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

import warnings
warnings.filterwarnings('ignore')


In [2]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


In [4]:
df = pd.read_csv(r"C:\Users\zizou\OneDrive\Desktop\stage 3ème\day 2\csvfiles\dataframefinale.csv",sep=";")

In [5]:
features = ["Nombre de Titres", "Echéance", "Taux"]
target = "Montant"

X = df[features]
y = df[target]


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [7]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42)
}


In [8]:
def evaluate_model(name, model):
    results = {}

    # 1) RMSE brut
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results["RMSE brut"] = rmse(y_test, preds)

    # 2) Normalisation
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    results["RMSE normalisé"] = rmse(y_test, preds)

    # 3) Feature Selection
    selector = SelectKBest(score_func=f_regression, k=2)
    X_train_sel = selector.fit_transform(X_train_scaled, y_train)
    X_test_sel = selector.transform(X_test_scaled)
    model.fit(X_train_sel, y_train)
    preds = model.predict(X_test_sel)
    results["RMSE feat. sel."] = rmse(y_test, preds)

    # 4) Hyperparameter tuning
    param_grid = {}
    if name == "Ridge Regression":
        param_grid = {"alpha": [0.1, 1, 10, 50]}
    elif name == "Lasso Regression":
        param_grid = {"alpha": [0.001, 0.01, 0.1, 1, 10]}
    elif name == "Decision Tree":
        param_grid = {"max_depth": [None, 5, 10, 20]}
    elif name == "Random Forest":
        param_grid = {"n_estimators": [50, 100, 200], "max_depth": [None, 10, 20]}
    elif name == "Gradient Boosting":
        param_grid = {"n_estimators": [50, 100, 200], "learning_rate": [0.01, 0.1, 0.2]}

    if param_grid:
        grid = GridSearchCV(model, param_grid, cv=3, scoring="neg_root_mean_squared_error", n_jobs=-1)
        grid.fit(X_train_sel, y_train)
        best_model = grid.best_estimator_
        preds = best_model.predict(X_test_sel)
        results["RMSE tuning"] = rmse(y_test, preds)
    else:
        results["RMSE tuning"] = results["RMSE feat. sel."]

    return results


In [9]:
all_results = {}
for name, model in models.items():
    print(f"Évaluation de {name} ...")
    all_results[name] = evaluate_model(name, model)


Évaluation de Linear Regression ...
Évaluation de Ridge Regression ...
Évaluation de Lasso Regression ...
Évaluation de Decision Tree ...
Évaluation de Random Forest ...
Évaluation de Gradient Boosting ...


In [10]:
results_df = pd.DataFrame(all_results).T
results_df


,RMSE brut,RMSE normalisé,RMSE feat. sel.,RMSE tuning
Linear Regression,7.025263,7.025263,7.024710,7.024710
Ridge Regression,7.025262,7.025256,7.024704,7.024648
Lasso Regression,7.024746,7.080449,7.080449,7.024677
Decision Tree,4.939495,4.939052,4.738220,4.709170
Random Forest,3.829547,3.814942,4.001083,3.953010
Gradient Boosting,4.614401,4.614401,4.723967,4.072807


In [11]:
results_df.sort_values("RMSE tuning")


,RMSE brut,RMSE normalisé,RMSE feat. sel.,RMSE tuning
Random Forest,3.829547,3.814942,4.001083,3.953010
Gradient Boosting,4.614401,4.614401,4.723967,4.072807
Decision Tree,4.939495,4.939052,4.738220,4.709170
Ridge Regression,7.025262,7.025256,7.024704,7.024648
Lasso Regression,7.024746,7.080449,7.080449,7.024677
Linear Regression,7.025263,7.025263,7.024710,7.024710
